In [1]:
%pip install --upgrade transformers torch scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.7.1
    Uninstalling scikit-learn-1.7.1:
      Successfully uninstalled scikit-learn-1.7.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import json
from dataclasses import dataclass, asdict
from typing import Optional

c:\Users\Abcom\anaconda3\envs\training\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@dataclass
class StructuredIntent:
    intent: str              # e.g. "kpi_query", "trend_analysis", "comparison", "unknown"
    metric: Optional[str]    # e.g. "revenue", "dau", "churn_rate"
    period: Optional[str]    # e.g. "last_quarter", "this_month", "ytd"
    granularity: Optional[str]  # e.g. "daily", "weekly", "monthly"
    filters: dict            # e.g. {"region": "APAC", "product": "Pro"}
    confidence: float

In [4]:
INTENT_LABELS = [
    "kpi_query",        # "What was revenue last quarter?"
    "trend_analysis",   # "Show me the trend of DAU over 6 months"
    "comparison",       # "Compare Q1 vs Q2 sales"
    "anomaly_detect",   # "Were there any spikes in churn last month?"
    "forecast",         # "Predict revenue for next quarter"
    "unknown",
]



In [5]:
class IntentClassifier:
    def __init__(self, model_name: str = "bert-base-uncased"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        # For a production system, fine-tune on your KPI queries.
        # For a quick start, use zero-shot classification:
        self.classifier = pipeline(
            "zero-shot-classification",
            model="facebook/bart-large-mnli",  # good zero-shot baseline
            device=0 if torch.cuda.is_available() else -1,
        )
        # NER pipeline to extract entities (metric names, time periods, filters)
        self.ner = pipeline(
            "token-classification",
            model="dslim/bert-base-NER",
            aggregation_strategy="simple",
        )

    def classify(self, query: str) -> StructuredIntent:
        # Step 1: Intent classification
        result = self.classifier(query, candidate_labels=INTENT_LABELS)
        top_intent = result["labels"][0]
        confidence = result["scores"][0]

        # Step 2: Entity extraction (metric, period, filters)
        entities = self._extract_entities(query)

        return StructuredIntent(
            intent=top_intent,
            metric=entities.get("metric"),
            period=entities.get("period"),
            granularity=entities.get("granularity"),
            filters=entities.get("filters", {}),
            confidence=confidence,
        )

    def _extract_entities(self, query: str) -> dict:
        """
        Rule-based + NER hybrid extraction.
        In production, replace/augment with a fine-tuned span extraction model.
        """
        query_lower = query.lower()
        entities = {"filters": {}}

        # --- Metric detection (extend this vocabulary) ---
        metric_keywords = {
            "revenue": ["revenue", "sales", "income", "arr", "mrr"],
            "dau": ["dau", "daily active users", "daily users"],
            "churn": ["churn", "churn rate", "attrition"],
            "conversion": ["conversion", "conversion rate", "cvr"],
            "ltv": ["ltv", "lifetime value", "customer lifetime"],
        }
        for metric, keywords in metric_keywords.items():
            if any(kw in query_lower for kw in keywords):
                entities["metric"] = metric
                break

        # --- Period detection ---
        period_keywords = {
            "last_quarter": ["last quarter", "previous quarter", "q-1"],
            "this_month":   ["this month", "current month", "mtd"],
            "last_month":   ["last month", "previous month"],
            "ytd":          ["ytd", "year to date", "this year"],
            "last_7_days":  ["last 7 days", "past week", "last week"],
        }
        for period, keywords in period_keywords.items():
            if any(kw in query_lower for kw in keywords):
                entities["period"] = period
                break

        # --- Granularity ---
        if "daily" in query_lower or "per day" in query_lower:
            entities["granularity"] = "daily"
        elif "weekly" in query_lower:
            entities["granularity"] = "weekly"
        elif "monthly" in query_lower:
            entities["granularity"] = "monthly"

        return entities

In [6]:
import pandas as pd
from pathlib import Path

# Load training data
data_dir = Path("../data")
training_data = pd.read_csv(data_dir / "supply_chain_train_full.csv")

print(f"Training data shape: {training_data.shape}")
print(f"\nColumns: {training_data.columns.tolist()}")
print(f"\nFirst 5 samples:")
print(training_data.head())
print(f"\nLabel distribution:")
print(training_data['label'].value_counts())

Training data shape: (350, 2)

Columns: ['text', 'label']

First 5 samples:
                                            text       label
0               What is the OTIF rate this week?  kpi_lookup
1  Show me on-time delivery percentage for today  kpi_lookup
2      What is the current failed delivery rate?  kpi_lookup
3  How many deliveries were completed yesterday?  kpi_lookup
4       What is the SLA breach count this month?  kpi_lookup

Label distribution:
label
kpi_lookup            50
trend_analysis        50
comparison            50
anomaly_detection     50
root_cause            50
forecast              50
operational_status    50
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split
from transformers import TextClassificationPipeline
import numpy as np

# Split training data into train and validation sets
train_dataset, val_dataset = train_test_split(
    training_data, 
    test_size=0.2, 
    random_state=42,
    stratify=training_data['label']
)

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

# Train the classifier
clf = IntentClassifier()

# For zero-shot classification, evaluate on training data
train_predictions = []
for text in train_dataset['text'].values:
    result = clf.classify(text)
    train_predictions.append(result.intent)

train_dataset['predicted'] = train_predictions

# Calculate training accuracy
accuracy = (train_dataset['label'] == train_dataset['predicted']).mean()
print(f"\nZero-shot Classifier Accuracy on Training Data: {accuracy:.2%}")

# Show some predictions
print("\nSample Predictions:")
print(train_dataset[['text', 'label', 'predicted']].head(10))

Training set size: 280
Validation set size: 70


c:\Users\Abcom\anaconda3\envs\training\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abcom\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Abcom\anaconda3\envs\training\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `hugg


Zero-shot Classifier Accuracy on Training Data: 27.14%

Sample Predictions:
                                                  text               label  \
243  What is driving the cost per kg increase for L...          root_cause   
194  Did truck utilisation fall below threshold for...   anomaly_detection   
250                        Predict OTIF for next month            forecast   
342  What is the real-time fill rate for trucks on ...  operational_status   
5                Show me the return rate for last week          kpi_lookup   
286   Estimate SLA breach count for the festive period            forecast   
223  What caused the increase in transit time for c...          root_cause   
79     How has cost per kg changed over the last year?      trend_analysis   
185  Was there unusual inbound shipment volume at w...   anomaly_detection   
64      How has last mile cost trended year over year?      trend_analysis   

      predicted  
243     unknown  
194  comparison  
250    for

In [ ]:
if __name__ == "__main__":
    clf = IntentClassifier()
    query = "What was our revenue growth last quarter broken down by region?"
    intent = clf.classify(query)
    print(json.dumps(asdict(intent), indent=2))